In [ ]:
%run "Belief States.ipynb"
import random
import time
import threading
import ipywidgets as widgets
from IPython.display import display, HTML

# --- CSS Styling ---
custom_css = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght=400;500;600;700&family=JetBrains+Mono&display=swap');

.app-container { background-color: #f9f9ff; font-family: 'Inter', sans-serif; }
.modern-card { background-color: #ffffff; border: 1px solid #c3c6d7; border-radius: 12px; padding: 20px; box-shadow: 0 1px 3px rgba(0,0,0,0.05); box-sizing: border-box; }
.card-header { font-size: 16px; font-weight: 600; color: #141b2b; border-bottom: 1px solid #c3c6d7; padding-bottom: 10px; margin-bottom: 15px; display: flex; justify-content: space-between; }

.stat-box { background-color: #ffffff !important; border: 1px solid #c3c6d7 !important; border-radius: 12px !important; padding: 15px !important; box-sizing: border-box; }

.puzzle-input input[type="number"] {
    font-size: 18px !important; font-weight: 700 !important; text-align: center !important; color: #004ac6 !important;
    background-color: #e1e8fd !important; border: 1px solid rgba(0,74,198,0.2) !important; border-radius: 8px !important;
    height: 100% !important; box-sizing: border-box;
}

.btn-primary { 
    background-color: #004ac6 !important; color: white !important; border-radius: 999px !important; 
    font-weight: 600 !important; border: 1px solid #004ac6 !important; width: 95% !important; box-sizing: border-box !important;
}
.btn-primary:hover { background-color: #003ea8 !important; }

.btn-action { background-color: #e1e8fd !important; color: #38485d !important; border-radius: 8px !important; font-weight: 600 !important; border: 1px solid rgba(0,74,198,0.2) !important; font-size: 11px !important; padding: 4px 8px !important; }

.log-output { background-color: #f1f3ff !important; font-family: 'JetBrains Mono', monospace !important; border: none !important; }

.breakdown-table {
    width: 100%;
    border-collapse: collapse;
    font-size: 12px;
    text-align: center;
}
.breakdown-table th {
    background-color: #f1f5f9;
    color: #475569;
    font-weight: 600;
    padding: 8px;
    border: 1px solid #e2e8f0;
}
.breakdown-table td {
    padding: 8px;
    border: 1px solid #e2e8f0;
    color: #0f172a;
}
.breakdown-table tr:nth-child(even) {
    background-color: #f8fafc;
}

.anim-board { display: grid; grid-template-columns: repeat(3, 1fr); gap: 8px; max-width: 160px; margin: 0 auto; background-color: #f1f3ff; padding: 12px; border-radius: 12px; }
.anim-tile { aspect-ratio: 1; background-color: #ffffff; border: 1px solid #c3c6d7; border-radius: 8px; display: flex; align-items: center; justify-content: center; font-size: 18px; font-weight: 700; color: #004ac6; box-shadow: 0 1px 3px rgba(0,0,0,0.05); transition: all 0.3s ease; }
.anim-tile-empty { aspect-ratio: 1; background-color: rgba(220, 226, 247, 0.4); border: 2px dashed #c3c6d7; border-radius: 8px; }
</style>
"""

display(HTML(custom_css))

header_html = widgets.HTML(value="""
<div style="display: flex; justify-content: space-between; align-items: center; padding: 15px 30px; background-color: #ffffff; border-bottom: 1px solid #c3c6d7; font-family: 'Inter', sans-serif;">
    <span style="font-size: 20px; font-weight: 700; color: #141b2b;">Belief States Space Calculator</span>
    <div style="color: #004ac6; font-weight: 700; border-bottom: 2px solid #004ac6; padding-bottom: 4px; font-size: 14px;">Belief States</div>
</div>
""")

# Grids for Initial States
input_boxes_1 = [widgets.BoundedIntText(value=v, min=0, max=8, layout=widgets.Layout(width='auto', height='45px')) 
                 for v in [1, 2, 3, 4, 0, 6, 7, 5, 8]]
for box in input_boxes_1: box.add_class('puzzle-input')
input_grid_1 = widgets.GridBox(input_boxes_1, layout=widgets.Layout(grid_template_columns="repeat(3, 1fr)", gap="5px"))

input_boxes_2 = [widgets.BoundedIntText(value=v, min=0, max=8, layout=widgets.Layout(width='auto', height='45px')) 
                 for v in [1, 2, 3, 5, 0, 4, 6, 7, 8]]
for box in input_boxes_2: box.add_class('puzzle-input')
input_grid_2 = widgets.GridBox(input_boxes_2, layout=widgets.Layout(grid_template_columns="repeat(3, 1fr)", gap="5px"))

# Grids for Goal States
goal_boxes_1 = [widgets.BoundedIntText(value=v, min=0, max=8, layout=widgets.Layout(width='auto', height='45px')) 
                for v in [1, 2, 3, 4, 5, 6, 7, 8, 0]]
for box in goal_boxes_1: box.add_class('puzzle-input')
goal_grid_1 = widgets.GridBox(goal_boxes_1, layout=widgets.Layout(grid_template_columns="repeat(3, 1fr)", gap="5px"))

goal_boxes_2 = [widgets.BoundedIntText(value=v, min=0, max=8, layout=widgets.Layout(width='auto', height='45px')) 
                for v in [1, 2, 3, 4, 5, 6, 7, 8, 0]]
for box in goal_boxes_2: box.add_class('puzzle-input')
goal_grid_2 = widgets.GridBox(goal_boxes_2, layout=widgets.Layout(grid_template_columns="repeat(3, 1fr)", gap="5px"))

btn_random = widgets.Button(description="Random Start", layout=widgets.Layout(flex='1'))
btn_random.add_class('btn-action')

btn_reset = widgets.Button(description="Reset", layout=widgets.Layout(flex='1'))
btn_reset.add_class('btn-action')

btn_load_ex1 = widgets.Button(description="Load Bài Tập (Không giải được)", layout=widgets.Layout(flex='1'))
btn_load_ex1.add_class('btn-action')

btn_load_ex2 = widgets.Button(description="Load Ví dụ Giải được", layout=widgets.Layout(flex='1'))
btn_load_ex2.add_class('btn-action')

action_btns_1 = widgets.HBox([btn_random, btn_reset], layout=widgets.Layout(gap='5px', margin='10px 0 5px 0'))
action_btns_2 = widgets.HBox([btn_load_ex1, btn_load_ex2], layout=widgets.Layout(gap='5px'))

initial_state_card = widgets.VBox([
    widgets.HTML('<div class="card-header"><span>Trạng Thái Ban Đầu (Initial Belief State)</span></div>'),
    widgets.HBox([
        widgets.VBox([widgets.HTML('<span style="font-size:12px; font-weight:600; color:#54647a; display:block; margin-bottom:5px;">Ma Trận M_A</span>'), input_grid_1], layout=widgets.Layout(flex='1')),
        widgets.VBox([widgets.HTML('<span style="font-size:12px; font-weight:600; color:#54647a; display:block; margin-bottom:5px;">Ma Trận M_B</span>'), input_grid_2], layout=widgets.Layout(flex='1'))
    ], layout=widgets.Layout(gap='15px')),
    action_btns_1, action_btns_2
], layout=widgets.Layout(margin='0 0 20px 0'))
initial_state_card.add_class('modern-card')

slider_depth = widgets.IntSlider(value=6, min=1, max=12, step=1, description='Độ sâu tối đa:', layout=widgets.Layout(width='95%'))
input_limit = widgets.BoundedIntText(value=5000, min=100, max=200000, step=100, description='Giới hạn nút:', layout=widgets.Layout(width='95%'))
btn_calculate = widgets.Button(description="Tính toán không gian", layout=widgets.Layout(height='45px'))
btn_calculate.add_class('btn-primary')

config_card = widgets.VBox([
    widgets.HTML('<div class="card-header"><span>Trạng Thái Đích (Goal Belief State) & Cấu Hình</span></div>'),
    widgets.HBox([
        widgets.VBox([widgets.HTML('<span style="font-size:12px; font-weight:600; color:#54647a; display:block; margin-bottom:5px;">Ma Trận Đích G_A</span>'), goal_grid_1], layout=widgets.Layout(flex='1')),
        widgets.VBox([widgets.HTML('<span style="font-size:12px; font-weight:600; color:#54647a; display:block; margin-bottom:5px;">Ma Trận Đích G_B</span>'), goal_grid_2], layout=widgets.Layout(flex='1'))
    ], layout=widgets.Layout(gap='15px')),
    widgets.VBox([
        slider_depth,
        input_limit,
        btn_calculate
    ], layout=widgets.Layout(margin='15px 0 0 0'))
], layout=widgets.Layout(margin='0 0 20px 0'))
config_card.add_class('modern-card')

step_label = widgets.HTML('<span style="background:#e1e8fd; padding: 4px 10px; border-radius: 6px; font-size: 12px; font-weight: 600; color:#004ac6;">Ready</span>')
sim_header = widgets.HBox([
    widgets.HTML('<span style="font-size: 16px; font-weight: 600; color: #141b2b;">Mô Phỏng Trực Quan (Simulation)</span>'),
    step_label
], layout=widgets.Layout(justify_content='space-between', border_bottom='1px solid #c3c6d7', padding='0 0 10px 0', margin='0 0 15px 0', width='100%'))

anim_html = widgets.HTML(value="")
sim_card = widgets.VBox([sim_header, anim_html], layout=widgets.Layout(flex='1'))
sim_card.add_class('modern-card')

left_col = widgets.VBox([initial_state_card, config_card, sim_card], layout=widgets.Layout(flex='1.8', min_width='60%'))

stat_total = widgets.HTML()
stat_size2 = widgets.HTML()
stat_size1 = widgets.HTML()
stat_goal_depth = widgets.HTML()

def update_stat(html_widget, value):
    html_widget.value = f'<div style="font-size: 18px; font-weight: 700; color: #141b2b; text-align: center; height: 30px; display: flex; align-items: center; justify-content: center;">{value}</div>'

update_stat(stat_total, "-")
update_stat(stat_size2, "-")
update_stat(stat_size1, "-")
update_stat(stat_goal_depth, "-")

def make_stat_box(title, html_widget):
    box = widgets.VBox([
        widgets.HTML(f'<span style="font-size: 10px; font-weight: 600; color: #54647a; text-transform: uppercase; display: block; text-align: center; width: 100%;">{title}</span>'),
        html_widget
    ], layout=widgets.Layout(width='100%', align_items='center'))
    box.add_class('stat-box')
    return box

stat_grid = widgets.GridBox([
    make_stat_box("Belief States đã duyệt", stat_total),
    make_stat_box("Nút chưa hội tụ (|b| = 2)", stat_size2),
    make_stat_box("Nút đã hội tụ (|b| = 1)", stat_size1),
    make_stat_box("Độ sâu đạt Đích (Goal)", stat_goal_depth)
], layout=widgets.Layout(grid_template_columns="1fr 1fr", gap="12px", margin="0 0 15px 0"))

breakdown_html = widgets.HTML(value='<div style="color: #54647a; font-style: italic; text-align: center; padding: 20px;">Bấm "Tính toán không gian" để xem kết quả phân tích.</div>')
breakdown_card = widgets.VBox([
    widgets.HTML('<div style="display:flex; align-items:center; justify-content:space-between; border-bottom: 1px solid #c3c6d7; padding: 10px 15px; background-color: #e1e8fd; border-radius: 12px 12px 0 0;"><span style="font-size: 12px; font-weight: 700; color: #141b2b; text-transform: uppercase; letter-spacing: 0.5px;">Bảng Phân Tích Độ Sâu Không Gian</span></div>'),
    widgets.VBox([breakdown_html], layout=widgets.Layout(padding='15px', overflow='auto', max_height='220px'))
], layout=widgets.Layout(border='1px solid #c3c6d7', border_radius='12px', flex='1', margin='0 0 15px 0'))

log_content = widgets.HTML(value='<div style="color: #54647a; font-style: italic; text-align: center; padding: 20px;">Kế hoạch chi tiết của từng bước sẽ hiển thị ở đây.</div>')
log_output_box = widgets.VBox([log_content], layout=widgets.Layout(padding='15px', overflow='auto', max_height='300px'))
log_output_box.add_class('log-output')
log_card = widgets.VBox([
    widgets.HTML('<div style="display:flex; align-items:center; justify-content:space-between; border-bottom: 1px solid #c3c6d7; padding: 12px 20px; background-color: #e1e8fd; border-radius: 12px 12px 0 0;"><span style="font-size: 13px; font-weight: 700; color: #141b2b; text-transform: uppercase; letter-spacing: 0.5px;">Execution Log</span><div style="display:flex; gap: 5px;"><span style="color:#54647a; font-size:16px;">📋</span><span style="color:#54647a; font-size:16px;">⬇️</span></div></div>'),
    log_output_box
], layout=widgets.Layout(border='1px solid #c3c6d7', border_radius='12px', flex='1'))

right_col = widgets.VBox([stat_grid, breakdown_card, log_card], layout=widgets.Layout(flex='1.2', min_width='350px'))

main_app = widgets.HBox([left_col, right_col], layout=widgets.Layout(padding='15px', gap='15px'))
display(header_html, main_app)

def render_boards(states):
    html_content = '<div style="display: flex; gap: 20px; justify-content: center; flex-wrap: wrap;">'
    for idx, state in enumerate(states):
        html_content += f'<div style="text-align: center;"><div style="font-weight: 600; font-size: 11px; color: #54647a; margin-bottom: 5px;">Bản Đồ {idx+1}</div>'
        html_content += '<div class="anim-board">'
        for val in state:
            if val == 0:
                html_content += '<div class="anim-tile-empty"></div>'
            else:
                html_content += f'<div class="anim-tile">{val}</div>'
        html_content += '</div></div>'
    html_content += '</div>'
    anim_html.value = html_content

def print_log_belief_state(step_title, action, belief_state):
    if action:
        action_html = f'<p style="margin-bottom: 8px; font-weight: 600; color: #141b2b;">Bước {step_title}: Di chuyển ô trống sang <span style="color: #004ac6;">{action}</span></p>'
    else:
        action_html = f'<p style="margin-bottom: 8px; font-weight: 600; color: #141b2b;">{step_title}</p>'
    log_html = '<div style="margin-bottom: 15px;">' + action_html
    log_html += '<div style="display: flex; gap: 15px; flex-wrap: wrap;">'
    for idx, state in enumerate(belief_state):
        state_str = ""
        for row_idx in range(0, 9, 3):
            row = state[row_idx:row_idx+3]
            state_str += "  " + "    ".join([str(x) if x != 0 else "[ ]" for x in row]) + "\n"
        log_html += f'<div style="background-color: #ffffff; padding: 12px; border-radius: 8px; border: 1px solid rgba(195, 198, 215, 0.5); display: inline-block; min-width: 120px; text-align: center;">'
        log_html += f'<div style="font-size: 11px; font-weight: 600; color: #54647a; margin-bottom: 5px; border-bottom: 1px dashed rgba(195,198,215,0.5); padding-bottom: 3px;">Bản đồ {idx+1}</div>'
        log_html += '<pre style="margin: 0; font-family: monospace; font-size: 13px; line-height: 1.4; color: #141b2b; text-align: left;">' + state_str + '</pre>'
        log_html += '</div>'
    log_html += '</div></div><div style="border-top: 1px solid rgba(195, 198, 215, 0.5); margin-bottom: 15px; width: 100%;"></div>'
    return log_html

def animate_path(start_states, path):
    render_boards(start_states)
    step_label.value = f'<span style="background:#e1e8fd; padding: 4px 10px; border-radius: 6px; font-size: 12px; font-weight: 600; color:#004ac6;">Step 0/{len(path)}</span>'
    time.sleep(1.2)
    
    for step_idx, (action, next_belief) in enumerate(path):
        render_boards(list(next_belief))
        step_label.value = f'<span style="background:#e1e8fd; padding: 4px 10px; border-radius: 6px; font-size: 12px; font-weight: 600; color:#004ac6;">Step {step_idx + 1}/{len(path)} - Di chuyển: {action}</span>'
        time.sleep(0.8)
        
    step_label.value = '<span style="background:#d1f4e0; padding: 4px 10px; border-radius: 6px; font-size: 12px; font-weight: 600; color:#0d6e35;">Finished ✅</span>'

def solve_and_render_space(b):
    breakdown_html.value = '<div style="text-align: center; padding: 20px;"><b style="color: #004ac6;">Đang tính toán không gian Belief States...</b></div>'
    log_content.value = '<div style="text-align: center; padding: 20px;">Đang tạo nhật ký thực thi...</div>'
    
    start_1 = [box.value for box in input_boxes_1]
    start_2 = [box.value for box in input_boxes_2]
    goal_1 = [box.value for box in goal_boxes_1]
    goal_2 = [box.value for box in goal_boxes_2]
    
    render_boards([start_1, start_2])
    
    start_time = time.time()
    res = calculate_belief_states(
        [start_1, start_2],
        [goal_1, goal_2],
        max_depth=slider_depth.value,
        max_states_limit=input_limit.value
    )
    end_time = time.time()
    
    elapsed_ms = int((end_time - start_time) * 1000)
    
    # Cập nhật kết quả thống kê
    update_stat(stat_total, f"{res['total_explored']:,}")
    update_stat(stat_size2, f"{res['size_counts'][2]:,}")
    update_stat(stat_size1, f"{res['size_counts'][1]:,}")
    
    goal_depth_str = str(res['goal_reached_depth']) if res['goal_reached_depth'] is not None else "-"
    if res['goal_reached_depth'] is not None:
        update_stat(stat_goal_depth, f'<span style="color: #0d6e35;">{goal_depth_str}</span>')
    else:
        update_stat(stat_goal_depth, f'<span style="color: #54647a;">Chưa đạt</span>')
        
    # Tạo bảng phân tích chi tiết
    tbl_html = '<table class="breakdown-table">'
    tbl_html += '<tr><th>Độ sâu (Depth)</th><th>Số nút chưa hội tụ (|b|=2)</th><th>Số nút đã hội tụ (|b|=1)</th><th>Tổng số nút</th></tr>'
    
    depth_keys = sorted(res['depth_stats'].keys())
    for d in depth_keys:
        count1 = res['depth_stats'][d][1]
        count2 = res['depth_stats'][d][2]
        total_at_d = count1 + count2
        
        row_style = ' style="background-color:#d1f4e0; font-weight:bold;"' if d == res['goal_reached_depth'] else ''
        
        tbl_html += f'<tr{row_style}>'
        tbl_html += f'<td>{d}</td>'
        tbl_html += f'<td>{count2}</td>'
        tbl_html += f'<td>{count1}</td>'
        tbl_html += f'<td>{total_at_d}</td>'
        tbl_html += '</tr>'
    tbl_html += '</table>'
    
    analysis_text = f'<div style="margin-top: 15px; font-size:12px; line-height: 1.5; color: #475569;">'
    analysis_text += f'<b>Thời gian tính toán:</b> {elapsed_ms} ms<br>'
    
    if res['unsolvable_states']:
        analysis_text += f'<span style="color:#c53030;">⚠️ <b>Phân tích Khả thi:</b> Phát hiện {len(res["unsolvable_states"])} trạng thái ban đầu có tính chẵn lẻ của số nghịch thế không khớp với bất kỳ trạng thái đích nào ({res["goal_parities"]}). Do đó, 2 ma trận này KHÔNG THỂ cùng đạt đích.</span><br>'
    else:
        analysis_text += f'<span style="color:#0d6e35;">✅ <b>Phân tích Khả thi:</b> Cả 2 trạng thái ban đầu đều cùng tính chẵn lẻ của số nghịch thế với ít nhất một trạng thái đích. Khả năng cao có thể tìm thấy kế hoạch Conformant để hội tụ về đích.</span><br>'
        
    if res['limit_exceeded']:
        analysis_text += f'<span style="color:#d97706;">⚠️ <b>Cảnh báo:</b> Đã đạt giới hạn nút tối đa ({input_limit.value:,}). Kết quả trên chỉ là một phần của không gian tìm kiếm.</span><br>'
        
    analysis_text += '</div>'
    
    breakdown_html.value = tbl_html + analysis_text
    
    # Tạo Execution Log
    if res['goal_path'] is not None:
        log_html_accum = ""
        log_html_accum += print_log_belief_state("Trạng thái bắt đầu", None, [start_1, start_2])
        for idx, (action, next_belief) in enumerate(res['goal_path']):
            log_html_accum += print_log_belief_state(str(idx + 1), action, list(next_belief))
        log_content.value = log_html_accum
        
        # Kích hoạt luồng mô phỏng trực quan
        threading.Thread(target=animate_path, args=([start_1, start_2], res['goal_path']), daemon=True).start()
    else:
        log_content.value = '<div style="color: #c53030; font-style: italic; text-align: center; padding: 20px;">Không tìm thấy kế hoạch đạt đích, không có Execution Log.</div>'
        step_label.value = '<span style="background:#fde8e8; padding: 4px 10px; border-radius: 6px; font-size: 12px; font-weight: 600; color:#c53030;">Không đạt đích</span>'

btn_calculate.on_click(solve_and_render_space)

def randomize_boards(b):
    nums1 = [1, 2, 3, 4, 5, 6, 7, 8, 0]
    random.shuffle(nums1)
    for i, box in enumerate(input_boxes_1): box.value = nums1[i]
    
    nums2 = [1, 2, 3, 4, 5, 6, 7, 8, 0]
    random.shuffle(nums2)
    for i, box in enumerate(input_boxes_2): box.value = nums2[i]
    render_boards([[box.value for box in input_boxes_1], [box.value for box in input_boxes_2]])
btn_random.on_click(randomize_boards)

def reset_boards(b):
    for box in input_boxes_1: box.value = 0
    for box in input_boxes_2: box.value = 0
    render_boards([[0]*9, [0]*9])
btn_reset.on_click(reset_boards)

def load_example_1(b):
    nums1 = [1, 2, 3, 4, 0, 6, 7, 5, 8]
    nums2 = [1, 2, 3, 5, 0, 4, 6, 7, 8]
    goal = [1, 2, 3, 4, 5, 6, 7, 8, 0]
    
    for i, box in enumerate(input_boxes_1): box.value = nums1[i]
    for i, box in enumerate(input_boxes_2): box.value = nums2[i]
    for i, box in enumerate(goal_boxes_1): box.value = goal[i]
    for i, box in enumerate(goal_boxes_2): box.value = goal[i]
    render_boards([nums1, nums2])
btn_load_ex1.on_click(load_example_1)

def load_example_2(b):
    nums1 = [1, 2, 3, 4, 5, 0, 7, 8, 6]
    nums2 = [1, 2, 3, 4, 5, 6, 7, 0, 8]
    goal = [1, 2, 3, 4, 5, 6, 7, 8, 0]
    
    for i, box in enumerate(input_boxes_1): box.value = nums1[i]
    for i, box in enumerate(input_boxes_2): box.value = nums2[i]
    for i, box in enumerate(goal_boxes_1): box.value = goal[i]
    for i, box in enumerate(goal_boxes_2): box.value = goal[i]
    render_boards([nums1, nums2])
btn_load_ex2.on_click(load_example_2)

# Khởi tạo mặc định: Load ví dụ giải được để tự động chạy
load_example_2(None)
solve_and_render_space(None)
